# 📱 ScrollMood
### Predicting University Student Satisfaction Levels Through Social Media Behavior Analysis

| | |
|---|---|
| **Model** | Random Forest Classifier |
| **Data Source** | Google Sheets (Live) |
| **Target** | Satisfaction Level — High / Medium / Low |
| **Split** | 70% Training / 30% Testing |
| **Responses** | 166 university students |


## 📦 Step 1 — Import Libraries

> Import all necessary libraries for data processing, machine learning, and visualization.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

print("✅ All libraries imported successfully!")

## 📂 Step 2 — Load Dataset

> Load responses directly from Google Sheets — no CSV download needed.
> Rename columns to short, readable names and drop the unused email column.

In [ ]:
## Load information from google sheet links wihout dowloading the csv file.
url = 'https://docs.google.com/spreadsheets/d/1SMpbQgOAdiQOjHFH4X5g2J56NWsrNT7L_BEfTsJR8zo/export?format=csv'
df = pd.read_csv(url)

## rename columns for better understanding.
df.columns = [
    'timestamp', 'email',
    'age', 'gender', 'year',
    'screen_time', 'platforms', 'content_type',
    'posting_freq', 'self_comparison', 'trend_pressure',
    'sleep_hours', 'sleep_affected', 'facetoface',
    'stress_after', 'sm_breaks',
    'social_satisfaction', 'academic_satisfaction',
    'overall_satisfaction', 'daily_happiness',
    'sm_impact'
]

## drop the (timestamp, email) as they are not useful for prdiction.
df.drop(columns=['timestamp', 'email'], inplace=True)

print(f"✅ Dataset loaded from Google Sheets!")
print(f"   Total responses : {df.shape[0]}") ## rows
print(f"   Total columns   : {df.shape[1]}") ## columns
df.head()

## 🧹 Step 3 — Data Cleaning #1: Handle Missing Values

> Check for missing values across all columns.
> Fill missing **numerical** values with the **median** and missing **categorical** values with the **mode** (most frequent answer).

In [ ]:
print("Missing values BEFORE cleaning:")
print(df.isnull().sum()) ## check for missing values in each column

numerical_cols = ['social_satisfaction', 'academic_satisfaction', 'overall_satisfaction', 'daily_happiness']
categorical_cols = ['age', 'gender', 'year', 'screen_time', 'platforms', 'content_type',
                    'posting_freq', 'self_comparison', 'trend_pressure', 'sleep_hours',
                    'sleep_affected', 'facetoface', 'stress_after', 'sm_breaks', 'sm_impact']

for col in numerical_cols: ## fill numerical columns with "median" values
    df[col] = df[col].fillna(df[col].median())

for col in categorical_cols: ## fill categorical columns with "mode" values
    df[col] = df[col].fillna(df[col].mode()[0])

print("\nMissing values AFTER cleaning:")
print(df.isnull().sum())
print("\n✅ Data Cleaning #1 Complete!")

## 🧹 Step 4 — Data Cleaning #2: Remove Duplicates & Outliers

> **Duplicates** — Remove identical rows to avoid the model learning the same data twice.
> **Outliers** — Use the IQR method to remove extreme satisfaction scores that do not represent typical students.

In [ ]:
# Remove duplicate rows
before = df.shape[0] ## number of rows
df.drop_duplicates(inplace=True)
print(f"Duplicates removed : {before - df.shape[0]} rows")

# Remove outliers from numerical columns
score_cols = ['social_satisfaction', 'academic_satisfaction', 'overall_satisfaction', 'daily_happiness']

before_outlier = df.shape[0]
mask = pd.Series(True, index=df.index) #mark all rows True
for col in score_cols:
    Q1  = df[col].quantile(0.25) ## 25th percentile
    Q3  = df[col].quantile(0.75) ## 75th percentile
    IQR = Q3 - Q1 
    ## keep rows that are within the IQR range
    mask &= (df[col] >= Q1 - 1.5 * IQR) & (df[col] <= Q3 + 1.5 * IQR)
df = df[mask] #if true, keep; if false, remove the row

print(f"Outliers removed   : {before_outlier - df.shape[0]} rows")
print(f"Remaining rows     : {df.shape[0]}")
print("\n✅ Data Cleaning #2 Complete!")

## 🎯 Step 5 — Create Target Variable

> Average the 4 satisfaction scores (Q15–Q18) to create a single happiness score per student.
> Classify into 3 levels: **High** (7–10), **Medium** (4–6), **Low** (1–3) — this becomes the prediction target.

In [ ]:
df['avg_satisfaction'] = df[score_cols].mean(axis=1)

def classify_satisfaction(score):
    if score >= 7:
        return 'High'
    elif score >= 4:
        return 'Medium'
    else:
        return 'Low'

df['satisfaction_level'] = df['avg_satisfaction'].apply(classify_satisfaction)

print("✅ Target variable created!")
print("\nSatisfaction Level Distribution:")
print(df['satisfaction_level'].value_counts())

## 🔄 Step 6 — Data Transformation #1: Encode Categorical Data

> ML models only understand numbers, not text.
> **Label Encoding** converts ordered answers (e.g. Never → 1, Always → 5).
> **One-Hot Encoding** converts content type into separate binary columns (0 or 1).

In [ ]:
## Encode categorical variables to numbers for modeling.
encoding_maps = {
    'age': {'18–20 ปี': 1, '21–23 ปี': 2, '24–26 ปี': 3, '27 ปีขึ้นไป / 27 and above': 4},
    'gender': {'Male / ชาย': 0, 'Female / หญิง': 1, 'Prefer not to say / ไม่ระบุ': 2},
    'year': {'Year 1 / ปี 1': 1, 'Year 2 / ปี 2': 2, 'Year 3 / ปี 3': 3, 'Year 4 / ปี 4': 4, 'Postgraduate / ป.โท / ป.เอก': 5},
    'screen_time': {
        'Less than 1 hour / น้อยกว่า 1 ชั่วโมง': 1, '1–3 hours / 1–3 ชั่วโมง': 2,
        '3–5 hours / 3–5 ชั่วโมง': 3, 'More than 5 hours / มากกว่า 5 ชั่วโมง': 4
    },
    'posting_freq':    {'Never / ไม่เคยเลย': 1, 'Rarely / นานๆ ครั้ง': 2, 'Sometimes / บางครั้ง': 3, 'Often / บ่อยพอสมควร': 4, 'Always / เป็นประจำ': 5},
    'self_comparison': {'Never / ไม่เคยเลย': 1, 'Rarely / นานๆ ครั้ง': 2, 'Sometimes / บางครั้ง': 3, 'Often / บ่อยพอสมควร': 4, 'Always / เป็นประจำ': 5},
    'trend_pressure':  {'Never / ไม่เคยเลย': 1, 'Rarely / นานๆ ครั้ง': 2, 'Sometimes / บางครั้ง': 3, 'Often / บ่อยพอสมควร': 4, 'Always / รู้สึกแบบนี้เป็นประจำ': 5},
    'sleep_hours':     {'Less than 5 hours / น้อยกว่า 5 ชั่วโมง': 1, '5–6 hours / 5–6 ชั่วโมง': 2, '7–8 hours / 7–8 ชั่วโมง': 3, 'More than 8 hours / มากกว่า 8 ชั่วโมง': 4},
    'sleep_affected':  {'Never / ไม่เคยเลย': 1, 'Rarely / นานๆ ครั้ง': 2, 'Sometimes / บางครั้ง': 3, 'Often / บ่อยพอสมควร': 4, 'Always / เป็นแบบนี้เป็นประจำ': 5},
    'facetoface':      {'Never / แทบไม่ได้เจอเลย': 1, '1–2 times / 1–2 ครั้ง': 2, '3–4 times / 3–4 ครั้ง': 3, 'Every day / เจอทุกวัน': 4},
    'stress_after':    {'Never / ไม่เคยเลย': 1, 'Rarely / นานๆ ครั้ง': 2, 'Sometimes / บางครั้ง': 3, 'Often / บ่อยพอสมควร': 4, 'Always / รู้สึกแบบนี้เป็นประจำ': 5},
    'sm_breaks':       {'Never / ไม่เคยหยุดเลย': 1, 'Rarely / นานๆ ครั้ง': 2, 'Sometimes / บางครั้ง': 3, 'Often / หยุดบ่อยพอสมควร': 4, 'Always / หยุดเป็นประจำ': 5},
    'sm_impact': {
        'Very Negatively (1) / ส่งผลเสียมาก — ทำให้ไม่มีความสุขเลย': 1,
        'Negatively (2) / ส่งผลเสียบ้าง — ทำให้ไม่ค่อยมีความสุข': 2,
        'Neutral (3) / ไม่ได้ส่งผลอะไร — เฉยๆ': 3,
        'Positively (4) / ส่งผลดีบ้าง — ทำให้มีความสุขขึ้น': 4,
        'Very Positively (5) / ส่งผลดีมาก — ทำให้มีความสุขมากเลย': 5
    }
}

## Apply the encoding maps to the respective columns
for col, mapping in encoding_maps.items():
    df[col + '_enc'] = df[col].map(mapping)
    print(f"✅ Encoded: {col}")

## One-hot encode the 'content_type' column and concatenate with original dataframe
## for example, if content_type has values like "Entertainment", "Educational", "News"
# we will create new columns "content_Entertainment", "content_Educational", "content_News" with binary values (0 or 1) indicating the presence of that content type.
content_dummies = pd.get_dummies(df['content_type'], prefix='content')
df = pd.concat([df, content_dummies], axis=1)
print(f"\n✅ One-Hot Encoded: content_type → {list(content_dummies.columns)}")
print("\n✅ Data Transform #1 Complete!")

## 📏 Step 7 — Data Transformation #2: Feature Scaling (Min-Max Normalization)

> Scale all 19 features to a range of **0.0 – 1.0** using Min-Max Normalization.
> This ensures no single feature dominates the model just because it has larger numbers.

In [ ]:
feature_cols = [
    'age_enc', 'gender_enc', 'year_enc',
    'screen_time_enc', 'posting_freq_enc', 'self_comparison_enc',
    'trend_pressure_enc', 'sleep_hours_enc', 'sleep_affected_enc',
    'facetoface_enc', 'stress_after_enc', 'sm_breaks_enc', 'sm_impact_enc'
] + list(content_dummies.columns)

df.dropna(subset=feature_cols, inplace=True)

X = df[feature_cols].copy()
y = df['satisfaction_level']

scaler   = MinMaxScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=feature_cols)

print(f"✅ Min-Max Normalization applied to {len(feature_cols)} features")
print(f"   All values scaled to range 0.0 – 1.0")
print("\nSample of scaled data:")
X_scaled.head(3).round(3)

## ✂️ Step 8 — Split Data (70% Train / 30% Test)

> Split the dataset into **training set (70%)** — used to teach the model, and **testing set (30%)** — used to evaluate performance on unseen data.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.3, random_state=42, stratify=y
)

print(f"✅ Data split complete!")
print(f"   Training set : {X_train.shape[0]} students (70%)")
print(f"   Testing set  : {X_test.shape[0]} students (30%)")

## 🌲 Step 9 — Train Model: Random Forest Classifier

> Train the **Random Forest** model on the training set.
> It builds **100 decision trees**, each learning different patterns — then combines their votes for a final prediction.

In [ ]:
rf_model = RandomForestClassifier(
    n_estimators = 100,   # Number of decision trees
    max_depth    = 10,    # Maximum depth per tree
    random_state = 42     # Ensures same results every run
)
rf_model.fit(X_train, y_train)
print("✅ Random Forest model trained successfully with 100 decision trees!")

## 🎯 Step 10 — Evaluate Model & Show Predictions

> Run the model on the **test set** and measure its performance.
> **Accuracy** — overall correctness. **Classification Report** — precision and recall per class.

In [ ]:
y_pred   = rf_model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print(f"🎯 Model Accuracy: {accuracy * 100:.2f}%")
print("\n📊 Classification Report:")
print(classification_report(y_test, y_pred))

## 📊 Step 11 — Confusion Matrix

> Visualize how many predictions were **correct vs incorrect** for each satisfaction level.
> Diagonal cells = correct predictions. Off-diagonal = mistakes.

In [ ]:
cm = confusion_matrix(y_test, y_pred, labels=['Low', 'Medium', 'High'])
plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Low', 'Medium', 'High'],
            yticklabels=['Low', 'Medium', 'High'])
plt.title('ScrollMood — Confusion Matrix', fontsize=14, fontweight='bold')
plt.xlabel('Predicted Satisfaction Level')
plt.ylabel('Actual Satisfaction Level')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150)
plt.show()
print("✅ Saved as confusion_matrix.png")

## 🌟 Step 12 — Feature Importance

> Shows which social media habits **influenced the prediction the most**.
> Higher score = stronger impact on student satisfaction level.

In [ ]:
feat_df = pd.DataFrame({
    'Feature'   : feature_cols,
    'Importance': rf_model.feature_importances_
}).sort_values('Importance', ascending=False).head(10)

plt.figure(figsize=(9, 6))
sns.barplot(data=feat_df, x='Importance', y='Feature', palette='Blues_r')
plt.title('ScrollMood — Top 10 Feature Importances', fontsize=14, fontweight='bold')
plt.xlabel('Importance Score')
plt.ylabel('Feature')
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150)
plt.show()
print("✅ Saved as feature_importance.png")

## 👀 Step 13 — Sample Predictions

> Show the first 10 predictions — comparing what the model predicted vs the actual satisfaction level.

In [ ]:
results = pd.DataFrame({
    'Actual'    : list(y_test[:10]),
    'Predicted' : list(y_pred[:10]),
    'Correct ✅': ['✅' if a == p else '❌'
                   for a, p in zip(list(y_test[:10]), list(y_pred[:10]))]
})
print("Sample Predictions (First 10 Students):")
print(results.to_string(index=False))
print(f"\n🎯 Final Model Accuracy: {accuracy * 100:.2f}%")